In [1]:
import os
import cv2
import json
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A
import time
from tqdm import tqdm

## Multi-Class Dataset Setup

This notebook supports generating synthetic datasets for **one or more object classes** in a single run.

### Data Directory Structure

The **folder name is the class name**. Organise your raw clipped images into class-named subdirectories inside `clipped_images/` before running the preparation scripts:

```
data/
├── bg/                     # Background images (shared across all classes)
├── clipped_images/
│   ├── estop/              # Raw RGBA PNGs for "estop" class
│   └── button/             # Raw RGBA PNGs for "button" class
├── images/                 # Auto-populated by remove_transparency_samsung.py
│   ├── estop/
│   └── button/
└── masks/                  # Auto-populated by cropped_images_to_masks.py
    ├── estop/
    └── button/
```

### One-Shot Data Preparation (run once for all classes)

```bash
cd data
python remove_transparency_samsung.py  # clipped_images/*/ -> images/*/
python rotate_by_90_180_270.py         # adds rotated variants to images/*/
python cropped_images_to_masks.py      # images/*/ -> masks/*/
```

Then run all cells in this notebook to generate the dataset.

**Rules:**
- The folder name **is** the class name — no manual ID assignment needed.
- YOLO integer class IDs are assigned at label-writing time in sorted alphabetical order of class names.
- Each image in `images/<class>/` must have a matching mask in `masks/<class>/` with the **same filename**.
- To add a new class, add a `clipped_images/<new_class>/` directory and re-run the three preparation scripts.


In [ ]:
PATH_MAIN = "data"

# The folder name is the class name.
# Each subdirectory inside data/images/ is one class; the directory name is used directly
# as the class label throughout the pipeline.
# YOLO integer class IDs are derived from the sorted list of class names at label-writing time.

class_names = sorted([
    d for d in os.listdir(os.path.join(PATH_MAIN, 'images'))
    if os.path.isdir(os.path.join(PATH_MAIN, 'images', d))
])
print("Detected classes:", class_names)

# Dictionaries keyed by class name (folder name)
class_files_imgs = {}
class_files_masks = {}

for class_name in class_names:
    imgs = sorted(os.listdir(os.path.join(PATH_MAIN, 'images', class_name)))
    imgs = [os.path.join(PATH_MAIN, 'images', class_name, f) for f in imgs]
    masks = sorted(os.listdir(os.path.join(PATH_MAIN, 'masks', class_name)))
    masks = [os.path.join(PATH_MAIN, 'masks', class_name, f) for f in masks]
    class_files_imgs[class_name] = imgs
    class_files_masks[class_name] = masks
    print(f"Class '{class_name}': {len(imgs)} images, {len(masks)} masks")

files_bg_imgs = sorted(os.listdir(os.path.join(PATH_MAIN, 'bg')))
files_bg_imgs = [os.path.join(PATH_MAIN, 'bg', f) for f in files_bg_imgs]
print("\nThe first five files from the sorted list of background images:", files_bg_imgs[:5])


In [3]:
def get_img_and_mask(img_path, mask_path):

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    mask = cv2.imread(mask_path)
    mask = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)
    
    mask_b = mask[:,:,0] == 0 # This is boolean mask
    mask = mask_b.astype(np.uint8) # This is binary mask
    
    return img, mask

In [ ]:
def visualize_single_img(img, mask, title, draw_bboxes=False):

    xmin = np.min(np.where(mask)[1])
    xmax = np.max(np.where(mask)[1])
    ymin = np.min(np.where(mask)[0])
    ymax = np.max(np.where(mask)[0])
    bbox = np.array([xmin, ymin, xmax, ymax])

    if draw_bboxes:
        start_point = (bbox[0], bbox[1])
        end_point = (bbox[2], bbox[3])
        img = cv2.rectangle(img.copy(), start_point, end_point, (255,0,0), 2)

    plt.figure(figsize=(16,16))
    plt.title(title, fontsize=18)
    plt.imshow(img)


In [ ]:
# Preview a random object and its binary mask from the first class.

preview_class = class_names[0]
preview_idx = min(9, len(class_files_imgs[preview_class]) - 1)
img_path = class_files_imgs[preview_class][preview_idx]
mask_path = class_files_masks[preview_class][preview_idx]

img, mask = get_img_and_mask(img_path, mask_path)

print("Preview class:", preview_class)
print("Image file:", img_path)
print("Mask file:", mask_path)
print("\nShape of the image of the object:", img.shape)
print("Shape of the binary mask:", mask.shape)

fig, ax = plt.subplots(1, 2, figsize=(16, 7))
ax[0].imshow(img)
ax[0].set_title('Object', fontsize=18)
ax[1].imshow(mask)
ax[1].set_title('Binary mask', fontsize=18);


In [ ]:
visualize_single_img(img, mask, title="Object with bounding box", draw_bboxes=True)


In [7]:
def resize_img(img, desired_max, desired_min=None):
   
    h, w = img.shape[0], img.shape[1]
    
    longest, shortest = max(h, w), min(h, w)
    longest_new = desired_max
    if desired_min:
        shortest_new = desired_min
    else:
        shortest_new = int(shortest * (longest_new / longest))
    
    if h > w:
        h_new, w_new = longest_new, shortest_new
    else:
        h_new, w_new = shortest_new, longest_new
        
    transform_resize = A.Compose([
        A.Sequential([
        A.Resize(h_new, w_new, interpolation=1, always_apply=False, p=1)
        ], p=1)
    ])

    transformed = transform_resize(image=img)
    img_r = transformed["image"]
        
    return img_r

In [ ]:
# Preview how a random background image can be resized with resize_img()

img_bg_path = files_bg_imgs[min(9, len(files_bg_imgs) - 1)]
img_bg = cv2.imread(img_bg_path)
img_bg = cv2.cvtColor(img_bg, cv2.COLOR_BGR2RGB)

img_bg_resized_1 = resize_img(img_bg, desired_max=640, desired_min=None)
img_bg_resized_2 = resize_img(img_bg, desired_max=640, desired_min=480)

print("Shape of the original background image:", img_bg.shape)
print("Shape of the resized background image (desired_max=640, desired_min=None):", img_bg_resized_1.shape)
print("Shape of the resized background image (desired_max=640, desired_min=480):", img_bg_resized_2.shape)

fig, ax = plt.subplots(1, 2, figsize=(16, 7))
ax[0].imshow(img_bg_resized_1)
ax[0].set_title('Resized (desired_max=640, desired_min=None)', fontsize=18)
ax[1].imshow(img_bg_resized_2)
ax[1].set_title('Resized (desired_max=640, desired_min=480)', fontsize=18);


In [ ]:
def resize_transform_obj(img,
                         mask,
                         longest_min,
                         longest_max,
                         transforms=False):

    h, w = mask.shape[0], mask.shape[1]

    longest, shortest = max(h, w), min(h, w)
    longest_new = np.random.randint(longest_min, longest_max)
    shortest_new = int(shortest * (longest_new / longest))

    if h > w:
        h_new, w_new = longest_new, shortest_new
    else:
        h_new, w_new = shortest_new, longest_new

    transform_resize = A.Compose([A.Resize(h_new,
                                           w_new,
                                           interpolation=1,
                                           always_apply=False,
                                           p=1)])

    transformed_resized = transform_resize(image=img, mask=mask)
    img_t = transformed_resized["image"]
    mask_t = transformed_resized["mask"]

    if transforms:
        transformed = transforms(image=img_t, mask=mask_t)
        img_t = transformed["image"]
        mask_t = transformed["mask"]

    return img_t, mask_t

transforms_obj = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=(-0.1, 0.2),
                               contrast_limit=0.1,
                               brightness_by_max=True,
                               always_apply=False,
                               p=1)
])


In [ ]:
preview_class = class_names[0]
preview_idx = min(9, len(class_files_imgs[preview_class]) - 1)
img_path = class_files_imgs[preview_class][preview_idx]
mask_path = class_files_masks[preview_class][preview_idx]
img, mask = get_img_and_mask(img_path, mask_path)

img_t, mask_t = resize_transform_obj(img,
                                     mask,
                                     longest_min=200,
                                     longest_max=470,
                                     transforms=transforms_obj)

print("\nShape of the image of the transformed object:", img_t.shape)
print("Shape of the transformed binary mask:", img_t.shape)

fig, ax = plt.subplots(1, 2, figsize=(16, 7))
ax[0].imshow(img_t)
ax[0].set_title('Transformed object', fontsize=18)
ax[1].imshow(mask_t)
ax[1].set_title('Transformed binary mask', fontsize=18);


In [ ]:
visualize_single_img(img_t, mask_t, title="Transformed object with bounding box", draw_bboxes=True)


In [ ]:
def add_obj(img_comp, mask_comp, img, mask, x, y, idx):
    '''
    img_comp - composition of objects
    mask_comp - composition of objects` masks
    img - image of object
    mask - mask of object
    x, y - coordinates where top-left corner of img is placed
    Function returns img_comp in CV2 RGB format + mask_comp as a list
    '''
    h_comp, w_comp = img_comp.shape[0], img_comp.shape[1]
    h, w = img.shape[0], img.shape[1]

    # Ensure that the object fits entirely within the composition
    if x < 0:
        img = img[:, -x:]
        mask = mask[:, -x:]
        x = 0
    if y < 0:
        img = img[-y:, :]
        mask = mask[-y:, :]
        y = 0
    if x + w > w_comp:
        img = img[:, :w_comp - x]
        mask = mask[:, :w_comp - x]
    if y + h > h_comp:
        img = img[:h_comp - y, :]
        mask = mask[:h_comp - y, :]

    mask_b = mask == 1
    mask_rgb_b = np.stack([mask_b, mask_b, mask_b], axis=2)

    img_comp[y:y+h, x:x+w, :] = img_comp[y:y+h, x:x+w, :] * ~mask_rgb_b + (img * mask_rgb_b)
    mask_comp[y:y+h, x:x+w] = mask_comp[y:y+h, x:x+w] * ~mask_b + (idx * mask_b)

    return img_comp, mask_comp


In [ ]:
def visualize_composition(img_comp, bboxes_comp=None):

    if bboxes_comp:
        for bbox in bboxes_comp:
            start_point, end_point = tuple([bbox[0], bbox[1]]), tuple([bbox[2], bbox[3]])
            img_comp = cv2.rectangle(img_comp.copy(), start_point, end_point, (255,0,0), 2)

    plt.figure(figsize=(40,40))
    plt.imshow(img_comp)


In [ ]:
img_bg_path = files_bg_imgs[min(10, len(files_bg_imgs) - 1)]
img_bg = cv2.imread(img_bg_path)
img_bg = cv2.cvtColor(img_bg, cv2.COLOR_BGR2RGB)

h, w = img_bg.shape[0], img_bg.shape[1]
mask_comp = np.zeros((h,w), dtype=np.uint8)

print("Shape of img_bg:", img_bg.shape)
print("Shape of mask:", mask.shape)

img_comp, mask_comp = add_obj(img_bg,
                              mask_comp,
                              img,
                              mask,
                              x=100,
                              y=100,
                              idx=1)

fig, ax = plt.subplots(1, 2, figsize=(16, 7))
ax[0].imshow(img_comp)
ax[0].set_title('Composition', fontsize=18)
ax[1].imshow(mask_comp)
ax[1].set_title('Composition mask', fontsize=18);


In [ ]:
bboxes_comp = create_bboxes_from_mask_comp(mask_comp)
visualize_composition(img_comp, bboxes_comp)


In [ ]:

img_comp, mask_comp = add_obj(img_comp,
                              mask_comp,
                              img_t,
                              mask_t,
                              x=400,
                              y=250,
                              idx=2)

fig, ax = plt.subplots(1, 2, figsize=(16, 7))
ax[0].imshow(img_comp)
ax[0].set_title('Composition', fontsize=18)
ax[1].imshow(mask_comp)
ax[1].set_title('Composition mask', fontsize=18);


In [ ]:
bboxes_comp = create_bboxes_from_mask_comp(mask_comp)
visualize_composition(img_comp, bboxes_comp)


In [18]:
def check_overlapping(mask_comp, obj_areas, overlap_degree=0):
    obj_ids = np.unique(mask_comp).astype(np.uint8)[1:-1]
    masks = mask_comp == obj_ids[:, None, None]
    
    ok = True
    
    if len(np.unique(mask_comp)) != np.max(mask_comp) + 1:
        ok = False
        return ok
    
    for idx, mask in enumerate(masks):
        if np.count_nonzero(mask) / obj_areas[idx] < 1 - overlap_degree:
            ok = False
            break
            
    return ok

In [ ]:
def create_composition(img_comp_bg,
                       max_objs=15,
                       longest_min=300,
                       longest_max=700,
                       overlap_degree=0,
                       max_attempts_per_obj=10):

    img_comp = img_comp_bg.copy()
    h, w = img_comp.shape[0], img_comp.shape[1]
    mask_comp = np.zeros((h,w), dtype=np.uint8)

    obj_areas = []
    obj_class_names_comp = []
    num_objs = np.random.randint(max_objs) + 2

    i = 1

    for _ in range(1, num_objs):

        for _ in range(max_attempts_per_obj):

            # Pick a random class (by name) and a random image from that class
            class_name = np.random.choice(class_names)
            imgs = class_files_imgs[class_name]
            masks = class_files_masks[class_name]
            idx = np.random.randint(len(imgs))

            img_path = imgs[idx]
            mask_path = masks[idx]

            img, mask = get_img_and_mask(img_path, mask_path)

            img_t, mask_t = resize_transform_obj(img,
                                                mask,
                                                longest_min,
                                                longest_max,
                                                transforms=transforms_obj)
            x_max, y_max = img_comp.shape[1] - img_t.shape[1], img_comp.shape[0] - img_t.shape[0]
            x, y = np.random.randint(x_max), np.random.randint(y_max)

            if i == 1:
                img_comp, mask_comp = add_obj(img_comp,
                                             mask_comp,
                                             img_t,
                                             mask_t,
                                             x,
                                             y,
                                             i)
                obj_areas.append(np.count_nonzero(mask_t))
                obj_class_names_comp.append(class_name)
                i += 1
                break
            else:
                img_comp_prev, mask_comp_prev = img_comp.copy(), mask_comp.copy()
                img_comp, mask_comp = add_obj(img_comp,
                                             mask_comp,
                                             img_t,
                                             mask_t,
                                             x,
                                             y,
                                             i)
                ok = check_overlapping(mask_comp, obj_areas, overlap_degree)
                if ok:
                    obj_areas.append(np.count_nonzero(mask_t))
                    obj_class_names_comp.append(class_name)
                    i += 1
                    break
                else:
                    img_comp, mask_comp = img_comp_prev.copy(), mask_comp_prev.copy()

    return img_comp, mask_comp, obj_class_names_comp


In [20]:
def create_bboxes_from_mask_comp(mask_comp):
    
    height, width = mask_comp.shape[0], mask_comp.shape[1]
    
    obj_ids = np.unique(mask_comp)[1:]
    masks = mask_comp == obj_ids[:, None, None]

    bboxes_comp = []
    
    for i in range(len(obj_ids)):
        pos = np.where(masks[i])
        xmin = np.min(pos[1])
        xmax = np.max(pos[1])
        ymin = np.min(pos[0])
        ymax = np.max(pos[0])

        bboxes_comp.append(list(map(int, [xmin, ymin, xmax, ymax])))

    return bboxes_comp

In [ ]:
img_comp_bg = cv2.imread(files_bg_imgs[min(10, len(files_bg_imgs) - 1)])
img_comp_bg = cv2.cvtColor(img_comp_bg, cv2.COLOR_BGR2RGB)
img_comp, mask_comp, class_names_comp = create_composition(img_comp_bg,
                                                            max_objs=1,
                                                            overlap_degree=0,
                                                            max_attempts_per_obj=1,
                                                            longest_min=20,
                                                            longest_max=470)
bboxes_comp = create_bboxes_from_mask_comp(mask_comp)

print("Classes in composition:", class_names_comp)
visualize_composition(img_comp, bboxes_comp)


In [ ]:
def generate_dataset(imgs_number, folder, split='train'):
    time_start = time.time()
    for j in tqdm(range(imgs_number)):
        idx = np.random.randint(len(files_bg_imgs))
        img_comp_bg = cv2.imread(files_bg_imgs[idx])
        img_comp_bg = cv2.cvtColor(img_comp_bg, cv2.COLOR_BGR2RGB)

        img_comp, mask_comp, class_names_comp = create_composition(img_comp_bg, max_objs=1,
                                                                    overlap_degree=0,
                                                                    max_attempts_per_obj=1,
                                                                    longest_min=20,
                                                                    longest_max=470)
        bboxes_comp = create_bboxes_from_mask_comp(mask_comp)

        # Create subfolders for 'images' and 'annotations'
        image_folder = os.path.join(folder, 'images', split)
        annotation_folder = os.path.join(folder, 'annotations', split)

        os.makedirs(image_folder, exist_ok=True)
        os.makedirs(annotation_folder, exist_ok=True)

        img_comp = cv2.cvtColor(img_comp, cv2.COLOR_RGB2BGR)
        cv2.imwrite(os.path.join(image_folder, '{}.jpg').format(j), img_comp)

        # Store human-readable class names alongside bboxes
        annotations = {}
        annotations['bboxes'] = bboxes_comp
        annotations['class_names'] = class_names_comp
        with open(os.path.join(annotation_folder, '{}.json').format(j), 'w') as f:
            json.dump(annotations, f)

    time_end = time.time()
    time_total = round(time_end - time_start)
    time_per_img = round((time_end - time_start) / imgs_number, 1)

    print("Generation of {} synthetic images is completed. It took {} seconds, or {} seconds per image".format(imgs_number, time_total, time_per_img))
    print("Images are stored in '{}'".format(os.path.join(folder, split, 'images')))
    print("Annotations are stored in '{}'".format(os.path.join(folder, split, 'annotations')))


In [23]:
generate_dataset(4000, folder='dataset', split='train')
generate_dataset(800, folder='dataset', split='val')

100%|██████████| 4000/4000 [02:50<00:00, 23.44it/s]


Generation of 4000 synthetic images is completed. It took 171 seconds, or 0.0 seconds per image
Images are stored in 'dataset/train/images'
Annotations are stored in 'dataset/train/annotations'


100%|██████████| 800/800 [00:33<00:00, 23.95it/s]

Generation of 800 synthetic images is completed. It took 33 seconds, or 0.0 seconds per image
Images are stored in 'dataset/val/images'
Annotations are stored in 'dataset/val/annotations'


In [ ]:
def convert_bbox_to_yolo(size, box):
    dw = 1.0 / (size[1])
    dh = 1.0 / (size[0])
    w = box[2] - box[0]
    h = box[3] - box[1]
    x = box[0] + w / 2.0
    y = box[1] + h / 2.0
    x = round(x * dw, 6)
    w = round(w * dw, 6)
    y = round(y * dh, 6)
    h = round(h * dh, 6)
    return (x, y, w, h)


def write_labels(json_files, data="train"):
    for json_file in json_files:
        with open(f"dataset/annotations/{data}/{json_file}") as f:
            annotation_content = json.load(f)

        img = cv2.imread(f"dataset/images/{data}/{json_file.strip('.json')}.jpg")
        lines = []
        for bbox, cls_name in zip(annotation_content["bboxes"], annotation_content["class_names"]):
            # Convert class name to integer ID (sorted alphabetical order matches config.yaml)
            class_id = class_names.index(cls_name)
            yolo_bbox = convert_bbox_to_yolo((img.shape), bbox)
            lines.append(str(class_id) + " " + str(yolo_bbox).replace(',', '').strip('()'))

        with open(f"dataset/labels/{data}/{json_file.strip('.json')}.txt", "w") as f:
            f.write("\n".join(lines))


os.makedirs("dataset/labels/train", exist_ok=True)
json_files = os.listdir("dataset/annotations/train")
write_labels(json_files, "train")

os.makedirs("dataset/labels/val", exist_ok=True)
json_files = os.listdir("dataset/annotations/val")
write_labels(json_files, "val")


In [ ]:
names_str = "\n".join([f" {i}: {name}" for i, name in enumerate(class_names)])
config = f"""# Data\ntrain: images/train\nval: images/val\n\n# Classes\nnames:\n{names_str}\n"""

with open("dataset/config.yaml", 'w') as f:
    f.write(config)

print(config)


In [26]:
# Blur 30% images

import random


# Function to apply motion blur to an image
def apply_motion_blur(image, angle=0):
    kernel_size = random.randint(5, 20)  # Random kernel size between 5 and 20
    motion_blur_kernel = np.zeros((kernel_size, kernel_size))
    motion_blur_kernel[int((kernel_size-1)/2), :] = 1.0 / kernel_size
    return cv2.warpAffine(cv2.filter2D(image, -1, motion_blur_kernel), 
                          cv2.getRotationMatrix2D((kernel_size/2, kernel_size/2), angle, 1), image.shape[1::-1])


# Function to apply random JPEG compression to an image
def apply_random_jpeg_compression(image):
    quality = random.randint(1, 30)  # Random JPEG quality between 1 and 30
    _, encoded_image = cv2.imencode('.jpg', image, [int(cv2.IMWRITE_JPEG_QUALITY), quality])
    return cv2.imdecode(encoded_image, 1)


def modify_images(input_folder):
    # Specify the folder containing your images
    input_folder = input_folder + '/'

    # List all image filenames in the input folder
    all_images = os.listdir(input_folder)
    all_images = [input_folder + image for image in all_images]

    # Calculate the number of images to process (30% of total images)
    num_images_to_process = int(0.3 * len(all_images))
    random_images = random.sample(all_images, num_images_to_process)

    for random_image in random_images:
        image = cv2.imread(random_image)
        image = apply_motion_blur(image)
        image = apply_random_jpeg_compression(image)
        cv2.imwrite(random_image, image)


modify_images('dataset/images/train')
modify_images('dataset/images/val')

In [ ]:
# from ultralytics import YOLO

# # Load a model
# model = YOLO('yolov8n.yaml')  # build a new model from YAML
# model = YOLO('yolov8n.pt')  # load a pretrained model (recommended for training)
# model = YOLO('yolov8n.yaml').load('yolov8n.pt')  # build from YAML and transfer weights

# # Train the model
# results = model.train(data=f'{os.getcwd()}/dataset/config.yaml', epochs=200, imgsz=640)
